In [1]:
from pathlib import Path
import torch
import torch.nn as nn
from config import get_config,get_weights_file_path
from train import get_model, get_ds, run_validation , greedy_decode
from dataset import BilingualDataset, causal_mask

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

model_filename = get_weights_file_path(config, f"23")
state = torch.load(model_filename, map_location=device)
model.load_state_dict(state['model_state_dict'])

Using device: cpu
Max length of the source sentence 16
Max length of the target sentence 16


<All keys matched successfully>

In [3]:
score = run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: print(msg), 0, None, num_examples=100)

--------------------------------------------------------------------------------
SOURCE: this audience about a brand and a
TARGET: इस श्रोतागण से एक ऐसे ब्रांड और
PREDICTED: इस से एक ऐसे ब्रांड और
--------------------------------------------------------------------------------
SOURCE: was actually would just actually take
TARGET: वास्तव में
PREDICTED: वास्तव में
--------------------------------------------------------------------------------
SOURCE: i'd say over 70 of the highly successful
TARGET: मैं कहूंगा कि जिन बेहद सफल
PREDICTED: मैं कहूंगा कि जिन बेहद सफल
--------------------------------------------------------------------------------
SOURCE: feelings and it was very upsetting for
TARGET: भावनाओं को ठेस पहुंचाई है और यह मेरे लिए भी बहुत परेशान करने वाला था
PREDICTED: भावनाओं को है और यह मेरे लिए भी बहुत परेशान करने वाला था
--------------------------------------------------------------------------------
SOURCE: do is a series of voice notes which i
TARGET: करने जा रहा हूं वह वॉयस 

## Processing the Dataset

In [1]:
from datasets import load_dataset
raw = load_dataset('SimranAswani/updated-english-to-hindi','default', split='train')

In [2]:
raw

Dataset({
    features: ['video_id', 'English subtitles', 'Hindi subtitles', '__index_level_0__', 'english', 'colloquial_hindi'],
    num_rows: 9437
})

In [3]:
raw['English subtitles']

["time i was like you'll be very private",
 "if you're watching this on youtube right",
 'um',
 "and that's the thing that i continually",
 'to that level',
 'hard and from the scientific perspective',
 "o'clock but is",
 'not being able to internalize',
 'everybody asks themselves every day make',
 'i just needed to get out of there and it',
 "that maybe i don't love as much or not",
 "this um so i'm going to ask you this",
 'bartlett',
 'purpose',
 'and getting to the bottom of what your',
 "this analogy and i don't know if it's a",
 'you had about six million subscribers',
 'and some people find that sort of',
 'knock me back and to swerve that being',
 'rain or that contract or because of',
 'had you said something about your if',
 'times I hated food the most it was the',
 'clear intention to yourself and to',
 "why don't you",
 'something',
 'question she said do you miss all the',
 'this is',
 "I'm going to tell you everything that I",
 "future value and how rich you'll be in",


In [4]:
raw['Hindi subtitles']

['समय मैं यही सोचता रहा कि आप बहुत निजी होंगे',
 'यदि आप इसे यूट्यूब पर देख रहे हैं तो सही है',
 'उम',
 'और यही वह चीज़ है जो मैं लगातार',
 'उस स्तर तक जानते हैं,',
 'और वैज्ञानिक दृष्टिकोण से',
 'बजे यहां आना चाहिए था, लेकिन क्या',
 'आंतरिक करने में सक्षम नहीं होना',
 'हर किसी को खुद से पूछने की सलाह देता हूं, उन',
 'मुझे बस वहां से निकलने की जरूरत थी और मुझे ऐसा',
 'जो शायद मुझे उतना पसंद नहीं हैं या जिनके',
 'यह कहा, इसलिए मैं आपसे यह प्रश्न पूछने जा रहा हूं',
 'बार्टलेट हूं',
 'उद्देश्य',
 'और वास्तव में आपकी प्रेरणाएँ क्या हैं इसकी तह तक जा रहे हैं,',
 'इस सादृश्य का उपयोग करता हूं और मुझे नहीं पता कि यह एक',
 'आपके पास लगभग छह मिलियन ग्राहक थे,',
 'और कुछ लोगों को ऐसा लगता है।  शौक में लेखन में अपने व्यवसाय के',
 'दस्तक मुझे वापस न देने दूं और उस',
 'बारिश या उस अनुबंध के कारण या जिस',
 'आपने आपके बारे में कुछ कहा है अगर',
 'समय से नफरत करता था।  भोजन',
 'अपने आप को और',
 'आप हमारे जैसा क्यों नहीं',
 'कुछ और में काम करने की कोशिश कर सकता हूं,',
 'सवाल पूछा, उसने कहा कि क्या तुम्ह

In [5]:
ra = []
for i in raw['English subtitles']:
    if i != None:
        ra.append(i)
    else:
        break

In [6]:
for i in range(len(ra)):
    print(ra[i])

time i was like you'll be very private
if you're watching this on youtube right
um
and that's the thing that i continually
to that level
hard and from the scientific perspective
o'clock but is
not being able to internalize
everybody asks themselves every day make
i just needed to get out of there and it
that maybe i don't love as much or not
this um so i'm going to ask you this
bartlett
purpose
and getting to the bottom of what your
this analogy and i don't know if it's a
you had about six million subscribers
and some people find that sort of
knock me back and to swerve that being
rain or that contract or because of
had you said something about your if
times I hated food the most it was the
clear intention to yourself and to
why don't you
something
question she said do you miss all the
this is
I'm going to tell you everything that I
future value and how rich you'll be in
the stored information on how to get
things that i need to know about you
mean with me and because of the
tweet at p

In [7]:
ha = []
for i in raw['Hindi subtitles']:
    if i != None:
        ha.append(i)
    else:
        break

In [8]:
for i in range(len(ha)):
    print(ha[i])

समय मैं यही सोचता रहा कि आप बहुत निजी होंगे
यदि आप इसे यूट्यूब पर देख रहे हैं तो सही है
उम
और यही वह चीज़ है जो मैं लगातार
उस स्तर तक जानते हैं,
और वैज्ञानिक दृष्टिकोण से
बजे यहां आना चाहिए था, लेकिन क्या
आंतरिक करने में सक्षम नहीं होना
हर किसी को खुद से पूछने की सलाह देता हूं, उन
मुझे बस वहां से निकलने की जरूरत थी और मुझे ऐसा
जो शायद मुझे उतना पसंद नहीं हैं या जिनके
यह कहा, इसलिए मैं आपसे यह प्रश्न पूछने जा रहा हूं
बार्टलेट हूं
उद्देश्य
और वास्तव में आपकी प्रेरणाएँ क्या हैं इसकी तह तक जा रहे हैं,
इस सादृश्य का उपयोग करता हूं और मुझे नहीं पता कि यह एक
आपके पास लगभग छह मिलियन ग्राहक थे,
और कुछ लोगों को ऐसा लगता है।  शौक में लेखन में अपने व्यवसाय के
दस्तक मुझे वापस न देने दूं और उस
बारिश या उस अनुबंध के कारण या जिस
आपने आपके बारे में कुछ कहा है अगर
समय से नफरत करता था।  भोजन
अपने आप को और
आप हमारे जैसा क्यों नहीं
कुछ और में काम करने की कोशिश कर सकता हूं,
सवाल पूछा, उसने कहा कि क्या तुम्हें सारी
यही वह बात है जो
मैं आपको वह सब कुछ बताने जा रहा हूं जो मैंने
भविष्य का मूल्य और आप अपने जीवन 

In [9]:
len(ha), len(ra)

(9427, 9427)

In [10]:
dictionary = {'translation': []}
for i in range(0, 9427):
    dictionary['translation'].append({'en':ra[i],'hi':ha[i]})

In [ ]:
dictionary

{'translation': [{'en': "time i was like you'll be very private",
   'hi': 'समय मैं यही सोचता रहा कि आप बहुत निजी होंगे'},
  {'en': "if you're watching this on youtube right",
   'hi': 'यदि आप इसे यूट्यूब पर देख रहे हैं तो सही है'},
  {'en': 'um', 'hi': 'उम'},
  {'en': "and that's the thing that i continually",
   'hi': 'और यही वह चीज़ है जो मैं लगातार'},
  {'en': 'to that level', 'hi': 'उस स्तर तक जानते हैं,'},
  {'en': 'hard and from the scientific perspective',
   'hi': 'और वैज्ञानिक दृष्टिकोण से'},
  {'en': "o'clock but is", 'hi': 'बजे यहां आना चाहिए था, लेकिन क्या'},
  {'en': 'not being able to internalize',
   'hi': 'आंतरिक करने में सक्षम नहीं होना'},
  {'en': 'everybody asks themselves every day make',
   'hi': 'हर किसी को खुद से पूछने की सलाह देता हूं, उन'},
  {'en': 'i just needed to get out of there and it',
   'hi': 'मुझे बस वहां से निकलने की जरूरत थी और मुझे ऐसा'},
  {'en': "that maybe i don't love as much or not",
   'hi': 'जो शायद मुझे उतना पसंद नहीं हैं या जिनके'},
  {'e

In [11]:
type(dictionary['translation'])

list

In [13]:
from datasets import Dataset
dataset = Dataset.from_dict(dictionary)

In [14]:
dataset

Dataset({
    features: ['translation'],
    num_rows: 9427
})

In [15]:
dataset.push_to_hub("Saurabh9901/english-hindi-dataset-2")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Saurabh9901/english-hindi-dataset-2/commit/41671103ba58310c62f765a4b3ff699b61c815dd', commit_message='Upload dataset', commit_description='', oid='41671103ba58310c62f765a4b3ff699b61c815dd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Saurabh9901/english-hindi-dataset-2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Saurabh9901/english-hindi-dataset-2'), pr_revision=None, pr_num=None)